# Mini Ignition Colab Demo

This notebook runs the Mini Ignition end-to-end demo in Google Colab.

Use one of these setup paths:

1. Set `REPO_URL` to a Git URL for your Mini Ignition repo.
2. Leave `REPO_URL` empty and upload a zip of the `mini-ignition` project folder when prompted.

The notebook installs the project, runs the test suite, runs `examples/demo.py`, and displays the generated hardware spec.

In [ ]:
# Optional: set this to your GitHub repo URL, for example:
# REPO_URL = "https://github.com/your-name/mini-ignition.git"
REPO_URL = ""

from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules

def find_project_dir(root: Path) -> Path | None:
    for pyproject in root.glob("**/pyproject.toml"):
        candidate = pyproject.parent
        if (candidate / "mini_ignition").is_dir():
            return candidate
    return None

current = Path.cwd()
if (current / "pyproject.toml").exists() and (current / "mini_ignition").is_dir():
    PROJECT_DIR = current
else:
    target = Path("/content/mini-ignition") if IN_COLAB else current / "mini-ignition"
    existing = find_project_dir(target) if target.exists() else None
    if existing is not None:
        PROJECT_DIR = existing
    elif REPO_URL:
        if target.exists():
            shutil.rmtree(target)
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
        PROJECT_DIR = target
    elif IN_COLAB:
        from google.colab import files

        print("Upload a zip file containing the mini-ignition project folder.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if not zip_names:
            raise RuntimeError("No .zip file uploaded. Upload mini-ignition.zip or set REPO_URL.")

        zip_path = Path("/content") / zip_names[0]
        zip_path.write_bytes(uploaded[zip_names[0]])
        extract_root = Path("/content/mini_ignition_upload")
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(extract_root)

        PROJECT_DIR = find_project_dir(extract_root)
        if PROJECT_DIR is None:
            raise RuntimeError("Could not find pyproject.toml and mini_ignition/ in the uploaded zip.")
    else:
        raise RuntimeError("Run this notebook from the project root, set REPO_URL, or use Colab upload.")

os.chdir(PROJECT_DIR)
print(f"Using project directory: {PROJECT_DIR}")

## Install Mini Ignition

This installs the project in editable mode so the demo can import `mini_ignition`.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

## Run The Test Suite

The full suite verifies the simulator, characterization, codegen, runtime, metrics, ladder, and controller.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest"], check=True)

## Run The End-to-End Demo

In [ ]:
demo = subprocess.run(
    [sys.executable, "examples/demo.py"],
    check=True,
    text=True,
    capture_output=True,
)
print(demo.stdout)

## Inspect The Generated Hardware Spec

In [ ]:
import json
from IPython.display import JSON, display

spec_path = Path("examples/generated_hw_spec.json")
print(f"Generated spec path: {spec_path.resolve()}")
display(JSON(json.loads(spec_path.read_text())))